## Sentinel-2 Level-1C stage-in and conversion to COG

This notebook uses the `stage-and-cog.1.0.0.cwl` CWL Workflow for staging-in and converting to COG of a Sentinel-2 Level-1C acquisition.

It uses the `Bash` kernel.

In [ ]:
cwltool /workspace/inference-eoap/cwl-workflow/stage-and-cog.cwl --help

Let's get an access token from the CDSE using a Python command-line tool:

In [ ]:
if [ "${EOAP_INFERENCE_LOCAL_FIXTURE:-0}" != 1 ] && [ -z "${CDSE_S3_ACCESS_KEY:-}" ]; then get-bearer-token --help; fi

Get the token:

In [ ]:
token=""
if [ "${EOAP_INFERENCE_LOCAL_FIXTURE:-0}" != 1 ] && [ -z "${CDSE_S3_ACCESS_KEY:-}" ]; then
    : "${CDSE_USERNAME:?Set CDSE_USERNAME for authenticated staging}"
    : "${CDSE_PASSWORD:?Set CDSE_PASSWORD for authenticated staging}"
    token=$(get-bearer-token --username "$CDSE_USERNAME" --password "$CDSE_PASSWORD")
fi

In [ ]:
echo "Staging mode: ${EOAP_INFERENCE_LOCAL_FIXTURE:-0} (1 uses the local fixture)"

Invoke the CWL Workflow:


In [ ]:
stac_url="https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l1c/items/S2A_MSIL1C_20240125T100311_N0510_R122_T33TVM_20240125T104959"

In [ ]:
export WORKSPACE=/workspace/inference-eoap
export RUNTIME=${WORKSPACE}/runs
mkdir -p "${RUNTIME}"
cd "${RUNTIME}"
if [ "${EOAP_INFERENCE_LOCAL_FIXTURE:-0}" = 1 ]; then
    python "${WORKSPACE}/scripts/create-fixture.py"
else
    STAC_URL="$stac_url" CDSE_TOKEN="$token" python - <<'PARAMS'
import json, os
with open("stage-params.json", "w") as stream:
    json.dump({"access_token": os.environ["CDSE_TOKEN"], "item": {"value": os.environ["STAC_URL"]}}, stream)
os.chmod("stage-params.json", 0o600)
PARAMS
    trap 'rm -f stage-params.json' EXIT
    cwltool --podman --preserve-environment CDSE_S3_ACCESS_KEY --preserve-environment CDSE_S3_SECRET_KEY --preserve-environment CDSE_S3_ENDPOINT --outdir "${RUNTIME}/staged" "${WORKSPACE}/cwl-workflow/stage-and-cog.cwl" stage-params.json > results.json
    rm -f stage-params.json
fi

Check the outcome

In [ ]:
cat ${WORKSPACE}/runs/results.json | jq .stac_catalog

We will use the path of the folder containing the STAC catalog.json file:

In [ ]:
cat ${WORKSPACE}/runs/results.json | jq -r .stac_catalog.path

In [ ]:
ls -l $( cat ${WORKSPACE}/runs/results.json | jq -r .stac_catalog.path )/catalog.json